# Linear Regression for a Laser Cutting Process - Solutions

In this exercise, we build our first models for the **laser cutting process** from the EDA exercise of session 1.
We fit linear regression models with the closed-form solution from the lecture and implement it ourselves with NumPy.

The production department has two questions which such a model might be able to answer:

1. **Planning:** How long does it take to produce a part at a given cutting speed (``ProductionTime``)?
2. **Quality:** How do the process parameters influence the burr height (``Gth_micro``)? The EDA ended with the hypothesis that a higher cutting speed and a lower gas pressure increase the burr height.

We use the file ``laser_cutting_clean.csv`` in the directory ``4_data``. It contains the data after the preprocessing of the EDA exercise:
duplicates removed, sensor dropouts (``999``) replaced by missing values, burr heights recorded in mm corrected to µm, and parts without burr height removed.
The four parts cut during a gas supply failure are still contained (valid outliers).

Questions in the tasks are meant for thinking - you do not have to write down your answers. Tasks marked with **Experiment** ask you to change your code and to observe the effect.

Tasks marked with **Optional** go one step further. Skip them if you are short on time - they are not needed for the following tasks.

The NumPy tutorial of this session covers all NumPy functions needed in this exercise. Do not be shy of using the [NumPy documentation](https://numpy.org/doc/stable/)!

### Contents <a id="contents"></a>
1. [From Data to Model](#1-from-data-to-model)
2. [Simple Linear Regression: Production Time](#2-simple-linear-regression)
    * [2.1 Closed-Form Solution](#2-1-closed-form-solution)
    * [2.2 Residuals and Sum of Squared Errors](#2-2-residuals-and-sse)
    * [2.3 Transforming the Feature](#2-3-transforming-the-feature)
    * [2.4 How Much Data Do We Need? (Optional)](#2-4-how-much-data)
3. [Multiple Linear Regression: Burr Height](#3-multiple-linear-regression)
    * [3.1 Normal Equation](#3-1-normal-equation)
    * [3.2 Burr Height Model for Steel](#3-2-burr-height-model-for-steel)
    * [3.3 Simple vs. Multiple Regression](#3-3-simple-vs-multiple-regression)
    * [3.4 Influence of the Data Preprocessing](#3-4-influence-of-the-data-preprocessing)
    * [3.5 All Materials (Optional)](#3-5-all-materials)
4. [Modeling Checklist](#4-modeling-checklist)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

## 1. From Data to Model <a id="1-from-data-to-model"></a>

[Back to top](#contents)

Before fitting a model, clarify what the model is used for. For each of the two questions of the production department, answer:

* What is the **target**, which attributes can be used as **features**?
* Which **type of learning** (supervised, unsupervised, reinforcement) and which **task** (regression, classification) is it?
* Is the main goal **prediction** or **inference**?

#### Explanation

| | Question 1: Planning | Question 2: Quality |
|---|---|---|
| Target | ``ProductionTime`` | ``Gth_micro`` (burr height) |
| Features | ``CuttingSpeed_mm_s`` (known before production) | process parameters, e.g., ``Thickness_mm``, ``CuttingSpeed_mm_s``, ``LaserPower_W``, ``GasPressure_bar``, ``Material`` |
| Type of learning | supervised (measured times are the labels) | supervised (measured burr heights are the labels) |
| Task | regression (numerical target) | regression (numerical target) |
| Main goal | **prediction**: production time of future parts | **inference**: understand how the parameters influence the burr height, check the EDA hypothesis |

Both goals are not exclusive: a model that describes the relationships well usually also predicts well - and vice versa, we want to understand why a prediction model works.

Read ``laser_cutting_clean.csv`` into a ``DataFrame`` named ``laser``, print its ``shape`` and the number of missing values per attribute.

In [ ]:
### BEGIN SOLUTION
laser = pd.read_csv('../4_data/laser_cutting_clean.csv')
print(laser.shape)
laser.isna().sum()
### END SOLUTION

## 2. Simple Linear Regression: Production Time <a id="2-simple-linear-regression"></a>

[Back to top](#contents)

The production department confirmed that ``ProductionTime`` is measured in **seconds** per part (an open question of the EDA).

We start with the simplest model: the production time $y$ as a linear function of the cutting speed $x$:

$$
\hat{y} = w_0 + w_1 \cdot x
$$

Create a ``DataFrame`` ``laser_time`` without the parts with a missing ``ProductionTime``.
Then create two NumPy arrays of shape ``(n,)``: ``x`` with the cutting speeds and ``y`` with the production times.
Create a scatter plot of the production time over the cutting speed.

*Note:* As in session 1, use seaborn to create the plots in this notebook (e.g., ``sns.scatterplot()``, ``sns.lineplot()``). Seaborn draws into matplotlib figures, so matplotlib functions such as ``plt.axhline()``, ``plt.xlabel()`` or ``plt.show()`` can be used to adjust the plots.

In [ ]:
### BEGIN SOLUTION
laser_time = laser.dropna(subset=['ProductionTime'])
x = laser_time['CuttingSpeed_mm_s'].to_numpy()
y = laser_time['ProductionTime'].to_numpy()

sns.scatterplot(laser_time, x='CuttingSpeed_mm_s', y='ProductionTime')
plt.show()
### END SOLUTION

assert x.shape == y.shape == (len(laser_time),), "x and y should be 1D arrays with one value per part."
assert not np.isnan(y).any(), "y contains missing values. Did you forget to remove the rows with missing production times?"

### 2.1 Closed-Form Solution <a id="2-1-closed-form-solution"></a>

In the lecture, the parameters minimizing the sum of squared errors $S$ (called *residual sum of squares*, RSS, in the lecture) were derived by setting its partial derivatives to zero (closed-form solution).
The resulting **least squares estimates** are:

$$
w_1 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}, \qquad w_0 = \bar{y} - w_1 \bar{x}
$$

where $\bar{x}$ and $\bar{y}$ are the means of $x$ and $y$. (In the lecture slides, the parameters are called $\beta_0, \beta_1$ instead of $w_0, w_1$.)

Implement the function ``fit_simple_linear_regression(x, y)``, which returns the tuple ``(w0, w1)``.
Use NumPy operations on whole arrays instead of loops.

In [ ]:
def fit_simple_linear_regression(x, y):
    ### BEGIN SOLUTION
    x_mean = x.mean()
    y_mean = y.mean()
    w1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
    w0 = y_mean - w1 * x_mean
    return w0, w1
    ### END SOLUTION

The following cell tests your implementation with points on the line $y = 1 + 2x$.

In [ ]:
w0_test, w1_test = fit_simple_linear_regression(np.array([0.0, 1.0, 2.0, 3.0]), np.array([1.0, 3.0, 5.0, 7.0]))
assert np.isclose(w0_test, 1) and np.isclose(w1_test, 2), "The parameters of the line y = 1 + 2x were not found."

Fit the model to the production times and print the parameters ``w0`` and ``w1``.
Plot the regression line into the scatter plot.

*Hint:* Evaluate the model on evenly spaced cutting speeds created with ``np.linspace()`` and draw it with ``sns.lineplot()``.

In [ ]:
### BEGIN SOLUTION
w0, w1 = fit_simple_linear_regression(x, y)
print(f'w0 = {w0:.2f}, w1 = {w1:.3f}')

speed_grid = np.linspace(x.min(), x.max(), 100)
sns.scatterplot(laser_time, x='CuttingSpeed_mm_s', y='ProductionTime')
sns.lineplot(x=speed_grid, y=w0 + w1 * speed_grid, color='red', label='linear model')
plt.show()
### END SOLUTION

Predict the production times for cutting speeds of 5, 20 and 45 mm/s. Which of these predictions do you trust?

In [ ]:
### BEGIN SOLUTION
for speed in [5, 20, 45]:
    print(f'{speed} mm/s: {w0 + w1 * speed:.1f} s')
### END SOLUTION

#### Explanation

The model is $\hat{y} \approx 67.5 - 1.65 \cdot x$: each additional mm/s reduces the production time by about 1.65 s.

The prediction for 45 mm/s is **negative** (about -6.6 s) - physically impossible.
45 mm/s lies outside of the cutting speeds in the data (3 to 42 mm/s), so this prediction is an **extrapolation**.
But already within the data range, the straight line does not follow the points well: the relationship is curved.

### 2.2 Residuals and Sum of Squared Errors <a id="2-2-residuals-and-sse"></a>

Implement the function ``sum_of_squared_errors(y, y_pred)``, which returns the sum of squared errors $S = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$.
The ``assert`` at the beginning of the function protects against the shape pitfall shown in the NumPy tutorial.

In [ ]:
def sum_of_squared_errors(y, y_pred):
    assert y.shape == y_pred.shape, f"Shapes do not match: {y.shape} and {y_pred.shape}"
    ### BEGIN SOLUTION
    return np.sum((y - y_pred) ** 2)
    ### END SOLUTION

In [ ]:
assert np.isclose(sum_of_squared_errors(np.array([1.0, 2.0, 3.0]), np.array([1.0, 0.0, 4.0])), 5)

Compute the predictions ``y_pred`` of the model for all parts, the residuals $y_i - \hat{y}_i$ and the sum of squared errors.

Plot the residuals over the cutting speed with ``sns.scatterplot()`` and add a horizontal line at zero (``plt.axhline(0, color='red')``).
Are the residuals scattered randomly around zero, or do you see a pattern?

In [ ]:
### BEGIN SOLUTION
y_pred = w0 + w1 * x
residuals = y - y_pred
sse_linear = sum_of_squared_errors(y, y_pred)
print(f'SSE = {sse_linear:.0f}')

sns.scatterplot(x=x, y=residuals)
plt.axhline(0, color='red')
plt.xlabel('CuttingSpeed_mm_s')
plt.ylabel('Residual (s)')
plt.show()
### END SOLUTION

**Experiment:** Increase and decrease ``w1`` by 10 % (keep ``w0`` unchanged) and recompute the sum of squared errors. What do you observe?

In [ ]:
### BEGIN SOLUTION
for factor in [0.9, 1.0, 1.1]:
    print(f'w1 * {factor}: SSE = {sum_of_squared_errors(y, w0 + factor * w1 * x):.0f}')
### END SOLUTION

#### Explanation

The residuals show a clear **pattern**: they are mostly positive for low (below ~10 mm/s) and high (above ~25 mm/s) cutting speeds and negative in between.
The model is systematically wrong - the residuals of a suitable model scatter randomly around zero.

Any change of the parameters increases the sum of squared errors: the closed-form solution is the **minimum** of $S(w_0, w_1)$. Fitting a model is an optimization problem.
In the next exercise, we will search this minimum iteratively with gradient descent.

### 2.3 Transforming the Feature <a id="2-3-transforming-the-feature"></a>

Let us use some domain knowledge: the production time of a part consists of a fixed time $t_{fix}$ (e.g., loading, unloading, piercing) and the cutting time, which is the cut length $L$ divided by the cutting speed $v$:

$$
\text{ProductionTime} = t_{fix} + L \cdot \frac{1}{v}
$$

This relationship is not linear in $v$, but it is linear in $\frac{1}{v}$. Therefore, we use the **inverse cutting speed** $x' = \frac{1}{v}$ as feature:

$$
\hat{y} = w_0 + w_1 \cdot x'
$$

Fit this model with your function ``fit_simple_linear_regression()`` and store the parameters in ``w0_inv`` and ``w1_inv``.
Compute the sum of squared errors and compare it to the model of the previous section.

Plot both models into the scatter plot of production time over cutting speed. Plot the residuals of the new model as well.

In [ ]:
### BEGIN SOLUTION
x_inv = 1 / x
w0_inv, w1_inv = fit_simple_linear_regression(x_inv, y)
print(f'w0 = {w0_inv:.2f}, w1 = {w1_inv:.1f}')

y_pred_inv = w0_inv + w1_inv * x_inv
sse_inv = sum_of_squared_errors(y, y_pred_inv)
print(f'SSE: {sse_inv:.0f} (linear in speed: {sse_linear:.0f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(laser_time, x='CuttingSpeed_mm_s', y='ProductionTime', ax=axes[0])
sns.lineplot(x=speed_grid, y=w0 + w1 * speed_grid, color='red', label='linear in speed', ax=axes[0])
sns.lineplot(x=speed_grid, y=w0_inv + w1_inv / speed_grid, color='green', label='linear in 1 / speed', ax=axes[0])
sns.scatterplot(x=x, y=y - y_pred_inv, ax=axes[1])
axes[1].axhline(0, color='red')
axes[1].set_xlabel('CuttingSpeed_mm_s')
axes[1].set_ylabel('Residual (s)')
plt.show()
### END SOLUTION

Interpret the parameters ``w0_inv`` and ``w1_inv`` including their units. Predict the production times for 5, 20 and 45 mm/s again.

Why is this model still called a *linear* regression, although the fitted curve is not a straight line?

In [ ]:
### BEGIN SOLUTION
for speed in [5, 20, 45]:
    print(f'{speed} mm/s: {w0_inv + w1_inv / speed:.1f} s')
### END SOLUTION

#### Explanation

The sum of squared errors is about 5 times smaller (about 6,600 instead of 33,000), and the residuals no longer show a pattern.

* $w_0 \approx 12.9$ s is the estimated **fixed time** per part.
* $w_1 \approx 334$ has the unit s $\cdot$ mm/s = mm: it is the estimated **cut length** of a part. As the parts have different cut lengths, which are not contained in the data, the model only knows the average cut length - this explains a large part of the remaining residuals.

The predictions (about 79.7 s, 29.6 s and 20.4 s) are plausible. For 45 mm/s, we still extrapolate, but the model now follows the physical relationship.

The model is **linear in its parameters** $w_0$ and $w_1$ - this is what *linear* regression means. The features can be arbitrary transformations of the measured attributes.
Choosing suitable transformations based on domain knowledge (*feature engineering*) often improves a model more than any algorithm.

### 2.4 How Much Data Do We Need? (Optional) <a id="2-4-how-much-data"></a>

**Optional:** How precise are the parameter estimates if fewer parts are available?

For each sample size ``n`` in ``[10, 30, 100]``, repeat the following 200 times: draw a random sample of ``n`` parts from ``laser_time`` (``laser_time.sample(n)``), fit the model with the inverse cutting speed and store the parameters.
Print the mean and the standard deviation of the estimated ``w0`` and ``w1`` for each sample size.

In [ ]:
### BEGIN SOLUTION
for n in [10, 30, 100]:
    estimates = []
    for _ in range(200):
        sample = laser_time.sample(n)
        estimates.append(fit_simple_linear_regression(1 / sample['CuttingSpeed_mm_s'].to_numpy(), sample['ProductionTime'].to_numpy()))
    estimates = np.array(estimates)
    print(f'n = {n:3d}: w0 = {estimates[:, 0].mean():5.1f} +- {estimates[:, 0].std():4.1f} s, '
          f'w1 = {estimates[:, 1].mean():5.1f} +- {estimates[:, 1].std():4.1f} mm')
### END SOLUTION

#### Explanation

On average, the estimates are close to the estimates on all parts, but their spread decreases with the sample size (for ``w1`` from about 60 mm for 10 parts to about 15 mm for 100 parts, roughly proportional to $\frac{1}{\sqrt{n}}$).

With few data, the parameters - and all conclusions drawn from them - are uncertain. In statistics, this uncertainty is quantified with confidence intervals and hypothesis tests (compare the lecture: modeling in statistics).

## 3. Multiple Linear Regression: Burr Height <a id="3-multiple-linear-regression"></a>

[Back to top](#contents)

Now, we turn to the quality question. The burr height depends on several process parameters at the same time, so we use **multiple linear regression**.
In the EDA, the material showed a strong influence on the burr height. As categorical features have to be encoded first, we start with the **steel** parts only:

$$
\hat{y} = w_0 + w_1 \cdot \text{Thickness} + w_2 \cdot \text{CuttingSpeed} + w_3 \cdot \text{LaserPower} + w_4 \cdot \text{GasPressure}
$$

### 3.1 Normal Equation <a id="3-1-normal-equation"></a>

For several features, the closed-form solution is written with matrices. The **feature matrix** $X$ contains one row per part. Its first column contains only ones (for the intercept $w_0$), the other columns contain the features.
The parameter vector $\mathbf{w} = (w_0, w_1, \dots, w_p)$ minimizing the sum of squared errors is the solution of the linear equation system (the **normal equation**):

$$
X^\top X \, \mathbf{w} = X^\top \mathbf{y}
$$

For a single feature, this system consists of exactly the two normal equations derived in the lecture.

Implement the two functions:
* ``design_matrix(df, columns)``: returns the feature matrix $X$ of shape ``(n, len(columns) + 1)`` with a first column of ones and the values of the given columns of the ``DataFrame`` ``df``.
* ``fit_linear_regression(X, y)``: returns the parameter vector $\mathbf{w}$ of shape ``(len(columns) + 1,)`` by solving the normal equation with ``np.linalg.solve()``.

In [ ]:
def design_matrix(df, columns):
    ### BEGIN SOLUTION
    return np.column_stack([np.ones(len(df)), df[columns].to_numpy()])
    ### END SOLUTION


def fit_linear_regression(X, y):
    ### BEGIN SOLUTION
    return np.linalg.solve(X.T @ X, X.T @ y)
    ### END SOLUTION

The following test fits the production time model with the inverse cutting speed again. The result must be identical to the result of your function ``fit_simple_linear_regression()``.

In [ ]:
X_time = design_matrix(laser_time.assign(InvSpeed=1 / laser_time['CuttingSpeed_mm_s']), ['InvSpeed'])
assert X_time.shape == (len(laser_time), 2)
assert np.all(X_time[:, 0] == 1)
w_time = fit_linear_regression(X_time, y)
assert w_time.shape == (2,)
assert np.allclose(w_time, [w0_inv, w1_inv]), "The normal equation should give the same parameters as the simple formula."

### 3.2 Burr Height Model for Steel <a id="3-2-burr-height-model-for-steel"></a>

Create a ``DataFrame`` ``steel`` containing only the steel parts without missing values in the features ``features`` (defined below).
Build the feature matrix ``X_steel`` and the target vector ``y_steel`` (``Gth_micro``) and fit the model.

Display the parameters as a ``pd.Series`` with the index ``['Intercept'] + features``.
Compute the sum of squared errors and the **root mean squared error** $RMSE = \sqrt{S / n} = \sqrt{MSE}$. Here, $MSE = S / n$ is the **mean squared error**, i.e., the mean of the squared errors. Taking the root gives the error in the unit of the target: the RMSE is the typical deviation of a prediction in µm.

In [ ]:
features = ['Thickness_mm', 'CuttingSpeed_mm_s', 'LaserPower_W', 'GasPressure_bar']

### BEGIN SOLUTION
steel = laser[laser['Material'] == 'Steel'].dropna(subset=features)
X_steel = design_matrix(steel, features)
y_steel = steel['Gth_micro'].to_numpy()
w_steel = fit_linear_regression(X_steel, y_steel)

sse_steel = sum_of_squared_errors(y_steel, X_steel @ w_steel)
print(f'n = {len(steel)}, SSE = {sse_steel:.0f}, RMSE = {np.sqrt(sse_steel / len(steel)):.2f} um')
pd.Series(w_steel, index=['Intercept'] + features)
### END SOLUTION

In [ ]:
assert X_steel.shape == (len(steel), len(features) + 1), "X_steel should have one row per steel part and one column per parameter."
assert y_steel.shape == (len(steel),), "y_steel should be a 1D array with one value per steel part."
assert not np.isnan(X_steel).any(), "X_steel contains missing values. Did you forget to remove the rows with missing features?"

Interpret the parameters: How much does the burr height change if
* the thickness increases by 0.1 mm,
* the cutting speed increases by 1 mm/s,
* the laser power increases by 100 W,
* the gas pressure increases by 1 bar,

while all other parameters are kept constant? Is the hypothesis of the EDA supported? Does the intercept have a meaningful interpretation?

#### Explanation

| Change (all other parameters constant) | Change of the burr height |
|---|---|
| Thickness + 0.1 mm | about + 1.0 µm |
| Cutting speed + 1 mm/s | about + 0.5 µm |
| Laser power + 100 W | about - 0.4 µm |
| Gas pressure + 1 bar | about - 1.8 µm |

The hypothesis of the EDA is supported: a higher cutting speed increases the burr height, a higher gas pressure reduces it.
The typical deviation of a prediction is about 2.8 µm (burr heights of steel parts range from about 15 to 50 µm).

The intercept (about 26 µm) is the predicted burr height for a thickness of 0 mm, a speed of 0 mm/s, a power of 0 W and a gas pressure of 0 bar - far outside of the data, without physical meaning. It is still needed to position the model.

*Note:* The data is synthetic, so we know the true effects used to generate it: + 1.2 µm per 0.1 mm thickness, + 0.5 µm per mm/s, - 0.3 µm per 100 W, - 1.2 µm per bar.
Speed and power are estimated well - why the gas pressure effect deviates, we investigate in Section 3.4.

### 3.3 Simple vs. Multiple Regression <a id="3-3-simple-vs-multiple-regression"></a>

**a)** Fit a simple linear regression of the burr height of the steel parts on the cutting speed only. Compare its slope to the cutting speed parameter of the multiple regression from 3.2. Which model estimates the larger effect of the cutting speed?

In [ ]:
### BEGIN SOLUTION
w0_speed, w1_speed = fit_simple_linear_regression(steel['CuttingSpeed_mm_s'].to_numpy(), y_steel)
print(f'Speed effect: simple regression {w1_speed:.2f}, multiple regression {w_steel[2]:.2f} um per mm/s')
### END SOLUTION

#### Explanation

The simple regression estimates a smaller effect of the cutting speed (about 0.40 instead of 0.53 µm per mm/s). The optional tasks b) to d) explain why.

**Optional:** Find out why the two parameters differ:

**b)** The multiple regression also contains the laser power. According to its parameter, does a higher laser power increase or decrease the burr height?

**c)** How are the cutting speed and the laser power related for the steel parts? Compute their correlation (*Hint:* ``df['column_a'].corr(df['column_b'])``) and remember the EDA: How did the operators choose the cutting speed?

**d)** Put a) to c) together: Why does the simple regression estimate a smaller effect of the cutting speed?

In [ ]:
### BEGIN SOLUTION
print(f'Laser power parameter: {w_steel[3]:.4f} um per W')
print(f"Correlation speed - power: {steel['CuttingSpeed_mm_s'].corr(steel['LaserPower_W']):.2f}")
### END SOLUTION

#### Explanation

**b)** The laser power parameter is negative: a higher laser power *reduces* the burr height (about 0.36 µm per 100 W).

**c)** The correlation is about 0.73: the operators set the cutting speed depending on the laser power (compare the EDA). Fast parts were usually also cut with a high laser power.

**d)** In the simple regression, the speed parameter also absorbs the effect of the laser power: fast parts get a higher burr due to the speed, but at the same time a lower burr due to the higher power. Therefore, the effect of the speed is underestimated. The multiple regression separates the effects of both features - an individual simple regression for each feature ignores the dependence between the features (compare the lecture).

If two features are very strongly correlated (e.g., $|r| > 0.9$), their effects can hardly be separated and the parameters become unstable (**multicollinearity**). With 0.73, this is not yet the case.

### 3.4 Influence of the Data Preprocessing <a id="3-4-influence-of-the-data-preprocessing"></a>

**Experiment:** Fit the model for the steel parts two more times:

* on the **raw data** ``laser_cutting.csv`` (without the preprocessing of the EDA, only remove the parts with missing values in the features or the target),
* on the cleaned data **without the gas supply failure** (only parts with ``GasPressure_bar > 4``).

Compare the parameters and the RMSE of the three models in one table. What do you observe?

In [ ]:
### BEGIN SOLUTION
laser_raw = pd.read_csv('../4_data/laser_cutting.csv')
datasets = {
    'cleaned': steel,
    'raw': laser_raw[laser_raw['Material'] == 'Steel'].dropna(subset=features + ['Gth_micro']),
    'cleaned, no gas failure': steel[steel['GasPressure_bar'] > 4],
}
results = {}
for name, data in datasets.items():
    X_data = design_matrix(data, features)
    y_data = data['Gth_micro'].to_numpy()
    w_data = fit_linear_regression(X_data, y_data)
    rmse = np.sqrt(sum_of_squared_errors(y_data, X_data @ w_data) / len(data))
    results[name] = list(w_data) + [rmse]
pd.DataFrame(results, index=['Intercept'] + features + ['RMSE']).round(4)
### END SOLUTION

#### Explanation

* **Raw data:** The two burr heights recorded in mm (about 0.03 instead of about 30 µm) produce huge residuals. As the errors are squared, these few values strongly pull the model: the RMSE increases from about 2.8 to 4.6 µm and, e.g., the thickness effect decreases from about 9.8 to 8.8 µm per mm.
* **Without the gas supply failure:** A single part (2.3 bar, 50 µm) changes the gas pressure effect from about -1.8 to -1.3 µm per bar, much closer to the true effect of -1.2. As this point lies far outside the usual pressure range (6 to 10 bar), it has a strong leverage on the slope.

The gas supply failure is a valid outlier, but it belongs to a different operating regime (the molten material is not blown out of the kerf).
A linear model for the normal operation should be fitted without it - and it is only valid in the range of the data (6 to 10 bar). In the EDA, we postponed this decision to the modeling phase: this is the phase.

### 3.5 All Materials (Optional) <a id="3-5-all-materials"></a>

**Optional:** To use the parts of all materials, the categorical attribute ``Material`` must be converted into numbers.
``pd.get_dummies(df['Material'], drop_first=True, dtype=float)`` creates one column per material containing 1 for parts of this material and 0 otherwise.
The first material (``Aluminum``) is dropped, as it is already described by zeros in all other material columns (*reference category*).

Create a ``DataFrame`` ``all_parts`` with all parts with ``GasPressure_bar > 4`` and without missing features, and add the dummy columns (``pd.concat([..., ...], axis=1)``).
Fit two models: (a) with the four numerical features only and (b) with the numerical features and the dummy columns. Compare the parameters.

In [ ]:
### BEGIN SOLUTION
all_parts = laser[laser['GasPressure_bar'] > 4].dropna(subset=features)
dummies = pd.get_dummies(all_parts['Material'], drop_first=True, dtype=float)
all_parts = pd.concat([all_parts, dummies], axis=1)
y_all = all_parts['Gth_micro'].to_numpy()

w_numerical = fit_linear_regression(design_matrix(all_parts, features), y_all)
w_material = fit_linear_regression(design_matrix(all_parts, features + list(dummies.columns)), y_all)
parameter_names = ['Intercept'] + features + list(dummies.columns)
pd.DataFrame({'numerical features': pd.Series(w_numerical, index=parameter_names[:len(w_numerical)]),
              'with material': pd.Series(w_material, index=parameter_names)}).reindex(parameter_names).round(4)
### END SOLUTION

#### Explanation

Without the material, the parameters are misleading: the laser power even gets a **positive** effect, and the effect of the cutting speed is much smaller than for the steel model.
The material influences both the burr height and the process settings (e.g., copper is cut slowly and has the highest burr heights). If such an influence is missing in the model, its effect is wrongly attributed to the other features (*omitted variable*).

With the material columns, the parameters of the numerical features are close to those of the steel model. The material parameters are the differences in burr height to aluminum at otherwise equal settings (about + 4.7 µm for steel and + 12 µm for copper).
For inference, all relevant influences have to be included in the model.

---
## 4. Modeling Checklist <a id="4-modeling-checklist"></a>

[Back to top](#contents)

The following checklist summarizes the steps of this exercise. You can use it as a guideline when building a model for your own data.

| Step | Questions to answer | Used in this notebook |
|---|---|---|
| **Goal** | Target and features? Type of learning and task? Prediction or inference? | Section 1 |
| **Data** | EDA done and preprocessing documented? Which range and operating regime does the data cover? | ``laser_cutting_clean.csv``, Section 3.4 |
| **Model** | Which features? Transformations based on domain knowledge? Categorical features encoded? | $\frac{1}{v}$ (Section 2.3), ``pd.get_dummies()`` (Section 3.5) |
| **Fitting** | Closed-form solution or iterative optimization? | ``fit_simple_linear_regression()``, normal equation with ``np.linalg.solve()`` |
| **Checking** | Residuals without pattern? Error measures (SSE, RMSE) compared on the same data? | residual plots, ``sum_of_squared_errors()`` |
| **Interpretation** | Parameters with units? "All other features constant" realistic? Correlated or missing features? Intercept meaningful? Extrapolation? | Sections 2.3, 3.2, 3.3, 3.5 |